# 10 — 2단계 앙상블의 **배율 강건성** (STEP 25 마지막 관문)

## 묻는 것 하나

STEP 25 에서 크롭 뷰 앙상블이 전체 데이터(27,331행)에서 확인됐습니다:

| | macro-F1 | accuracy | 커버리지(오답 20%) |
|---|---:|---:|---:|
| 릴리스 `m2.5` 단독 | 0.5959 | 0.6677 | 62.0% |
| **3팔 앙상블** | **0.6254** | **0.7037** | **71.5%** |

**그런데 관문 4개 중 하나를 못 쟀습니다** — 배율 강건성입니다.
`f320` 단독은 배율 하락이 **35.1%** 였습니다 (`m2.5` 26.0%). 앙상블이 그걸
물려받으면 macro-F1 이 올라도 **못 씁니다** (촬영 거리에 그대로 노출).

VL01 에서는 앙상블이 24.6% → **27.3%** 로 `m2.5` 쪽에 붙었지만, VL01 은
STEP 23·25 에서 두 번 배신했습니다. 전체 데이터로 확인합니다.

## 학습이 **없습니다**

이미 학습된 체크포인트 3개에 교란만 걸어 추론합니다.
2,000장 × 7조건 × 3모델 = 42,000회 — T4 로 15~25분입니다.
(크롭 연결에 30~60분이 더 듭니다.)

## 붙일 것 (Add Input)

| 입력 | 왜 |
|---|---|
| `m2.5` 크롭 | ✅ |
| `f320` 크롭 | ✅ |
| 매니페스트 | ✅ |
| **STEP 16 release** | 릴리스 `m2.5` convnextv2_base |
| **STEP 23 노트북 Output** | `stage2_effnetv2_s_{m2.5,f320}_384_moderate` 2개 |

⚠️ STEP 23 출력을 **New Dataset (Private)** 으로 만들어 붙이세요.

## 돌리는 법

우측 상단 **[Save Version] → Save & Run All (Commit)**.

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
NAME   = "deeplearning_test"
# ⚠️ 브랜치를 "main" 으로 **못 박으면 안 됩니다.** 아래 reset --hard 가
#    작업 브랜치를 통째로 덮어써서, 방금 만든 코드가 사라진 채로 몇 시간을
#    돌게 됩니다. 이미 리포 안에서 돌고 있으면 **지금 브랜치를 그대로 씁니다.**
#    바꾸려면 환경변수:  export DOG_SKIN_BRANCH=main
# ★ 이 노트북이 사는 브랜치. **여기서 못 박지 않으면 "main" 을 받습니다.**
#    캐글/콜랩은 리포가 없는 상태로 시작해서 아래 _ROOT 탐색이 실패하고,
#    예전 기본값이 "main" 이었습니다. main 이 뒤처져 있으면 **셀은 최신인데
#    src/ 만 옛것**인 채로 돕니다 — 실제로 며칠 그랬습니다 (main 75445c0).
#    첫 셀은 그 상태에서도 "코드 버전 …" 을 태연히 찍습니다.
NB_BRANCH = "claude/dog-disease-diagnosis-model-1s6jtf"
BRANCH = os.environ.get("DOG_SKIN_BRANCH", "")
_cwd   = os.getcwd()

# ⚠️ "지금 리포 안인가" 를 **폴더 이름으로만** 보면 안 됩니다. 주피터에서
#    notebooks/*.ipynb 를 열면 cwd 가 `.../deeplearning_test/notebooks` 라
#    이름이 안 맞고, 그러면 **리포 안에 리포를 또 clone** 합니다
#    (실제로 런팟에서 .../notebooks/deeplearning_test 가 생겼습니다).
#    위로 거슬러 올라가며 **진짜 리포 루트**를 찾습니다.
_p = os.path.abspath(_cwd)
_ROOT = None
while True:
    if (os.path.isdir(os.path.join(_p, ".git"))
            and os.path.isfile(os.path.join(_p, "src", "env.py"))):
        _ROOT = _p
        break
    _up = os.path.dirname(_p)
    if _up == _p:
        break
    _p = _up

if _ROOT:
    DIR = _ROOT           # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
    if not BRANCH:
        BRANCH = subprocess.run(["git", "-C", DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                                capture_output=True, text=True).stdout.strip() or "main"
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

BRANCH = BRANCH or NB_BRANCH

# ⚠️ 예전엔 fetch/reset 을 **둘 다 check=False** 로 불렀습니다. 실패해도 조용히
#    넘어가서, 캐글 클론이 **지워진 커밋(75445c0)에 붙박인 채 며칠을 돌았습니다.**
#    src/ 를 아무리 고쳐 푸시해도 안 실렸고, 첫 셀은 "코드 버전 …" 을 태연히
#    찍었습니다. 그 줄을 믿을 수 없다는 게 제일 나빴습니다.
#    → 이제 실패하면 **말하고, 클론을 지우고 다시 받습니다.**
#    (Kaggle Persistence 를 'Files' 로 켜두면 /kaggle/working 이 살아남아
#     낡은 클론이 계속 재사용됩니다 — 그 경우에도 여기서 복구됩니다.)
def _git(*args, cwd=None):
    return subprocess.run(["git", *args], capture_output=True, text=True, cwd=cwd)


def _fresh_clone(dst, branch):
    import shutil as _sh
    _sh.rmtree(dst, ignore_errors=True)
    r = _git("clone", "-b", branch, "--depth", "1", REPO, dst)
    if r.returncode != 0:
        raise RuntimeError("git clone 실패:\n" + (r.stderr or "")[-800:])


_need_clone = not os.path.isdir(os.path.join(DIR, ".git"))
if not _need_clone:
    # shallow clone 이라 origin/<브랜치> 대신 FETCH_HEAD 로 맞춥니다
    # (히스토리가 갈리면 origin/<브랜치> 가 옛 커밋을 가리킨 채 남습니다)
    r = _git("-C", DIR, "fetch", "--depth", "1", "origin", BRANCH)
    if r.returncode != 0:
        print("⚠️ git fetch 실패 — 클론을 새로 받습니다\n   " + (r.stderr or "")[-300:])
        _need_clone = True
    else:
        r = _git("-C", DIR, "reset", "--hard", "FETCH_HEAD")
        if r.returncode != 0:
            print("⚠️ git reset 실패 — 클론을 새로 받습니다\n   " + (r.stderr or "")[-300:])
            _need_clone = True

if _need_clone:
    _fresh_clone(DIR, BRANCH)

# ★ 정말 최신인지 **확인**합니다. 위가 다 성공해도 여기서 한 번 더 봅니다 —
#   "최신이라고 믿었는데 아니었다" 가 이 프로젝트에서 가장 비쌌던 실패입니다.
_local = _git("-C", DIR, "rev-parse", "HEAD").stdout.strip()
_remote = _git("-C", DIR, "ls-remote", REPO, f"refs/heads/{BRANCH}").stdout.split()
_remote = _remote[0] if _remote else ""
if _remote and _local and not _remote.startswith(_local[:8]) and not _local.startswith(_remote[:8]):
    print("\n" + "!" * 66)
    print(f"🚨 코드가 최신이 아닙니다 — 로컬 {_local[:8]} / 원격 {_remote[:8]}")
    print("   클론을 지우고 다시 받습니다.")
    print("!" * 66 + "\n")
    _fresh_clone(DIR, BRANCH)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", _git("-C", DIR, "log", "--oneline", "-1").stdout.strip())
print("브랜치      :", BRANCH,
      f"(원격 {_remote[:8]})" if _remote else "(원격 확인 실패)")
if BRANCH != NB_BRANCH:
    print(f"⚠️ 이 노트북이 만들어진 브랜치({NB_BRANCH})가 아닙니다 —")
    print("   src/ 가 셀보다 뒤처져 있을 수 있습니다. 아래 [nb] 줄을 꼭 보세요.")

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
# albumentations 는 import 할 때마다 PyPI 에 버전 확인 요청을 보냅니다.
# Kaggle 은 외부 네트워크가 막혀 있어 타임아웃(2초)만 기다리다 끝납니다 — 꺼둡니다.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

# ⚠️ 임대 GPU 이미지(런팟 등)의 파이썬은 **externally managed** 입니다 (PEP 668).
#    그냥 설치하면 첫 시도가 통째로 거부돼서, 재시도 로직이 있어도 무서운
#    에러 덩어리가 먼저 찍힙니다. 처음부터 허용해두면 그 소음이 없습니다.
#    Colab/Kaggle 에는 이 제약이 없어서 이 변수는 무해합니다.
os.environ["PIP_BREAK_SYSTEM_PACKAGES"] = "1"
os.environ["UV_BREAK_SYSTEM_PACKAGES"] = "1"

# ⚠️ Colab/Kaggle 에는 numpy·pandas·sklearn 이 이미 있지만 **임대 GPU 이미지엔
#    torch 만 있는 경우가 많습니다** (런팟에서 `No module named 'pandas'` 로
#    막혔습니다). 그렇다고 매번 다 깔면 Colab 에서 버전이 흔들리므로
#    **없는 것만** 깝니다.
_NEED = {                       # import 이름 → pip 이름
    "numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "PIL": "Pillow",
    "sklearn": "scikit-learn", "cv2": "opencv-python-headless", "tqdm": "tqdm",
    "matplotlib": "matplotlib", "timm": "timm", "imagehash": "imagehash",
    "pytorch_grad_cam": "grad-cam", "albumentations": "albumentations",
}
import importlib.util as _ilu

_PKGS = [pip for mod, pip in _NEED.items() if _ilu.find_spec(mod) is None]
if _PKGS:
    print(f"[env] 없는 패키지 {len(_PKGS)}개를 깝니다: {_PKGS}")
else:
    print("[env] 필요한 패키지가 전부 있습니다 — 설치를 건너뜁니다")

# ⚠️ 일부 이미지(런팟 PyTorch 등)는 파이썬이 **externally managed** 라
#    (PEP 668) --system 설치를 거부합니다. Colab/Kaggle 에는 없는 문제라
#    처음엔 안 넣었다가 런팟에서 첫 셀이 바로 죽었습니다.
#    --break-system-packages 를 붙여 한 번 더 시도합니다.
def _install(args: list[str]) -> bool:
    return subprocess.run(args, check=False).returncode == 0


_ok = not _PKGS          # 깔 게 없으면 이미 성공입니다
if _PKGS and _install([sys.executable, "-m", "pip", "install", "-q", "uv"]):
    _base = [sys.executable, "-m", "uv", "pip", "install", "-q", "--system"]
    _ok = _install(_base + _PKGS)
    if not _ok:
        _ok = _install(_base + ["--break-system-packages"] + _PKGS)
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    _p = [sys.executable, "-m", "pip", "install", "-q"]
    if not _install(_p + _PKGS):
        _install(_p + ["--break-system-packages"] + _PKGS)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-09-04.5"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# 환경 판정이 이상하면(예: Kaggle 인데 colab 이라고 나오면) 근거를 봅니다
if E.env != "local":
    env.diagnose()

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


## 1. 모델 3개를 찾습니다 — **없으면 여기서 멈춥니다**

체크포인트가 하나라도 없으면 뒤에서 조용히 2팔이 되거나 엉뚱한 걸 고릅니다
(2026-09-05 에 실제로 알파벳 순으로 다른 모델을 골라 정확도가 조용히
0.6689 → 0.6267 이 됐습니다). 그래서 **먼저 다 있는지 확인하고 멈춥니다.**

In [ ]:
import sys
sys.path.insert(0, DIR)
import numpy as np, torch
from pathlib import Path
from src import crop, env, labels, models, robust, split, stages, train
from src.config import CFG, CLASSES

env.load_prepared()
env.require_gpu()
DEV = "cuda"

# 팔: (이름, 실험 폴더 이름, 크롭 태그).  **첫 번째가 기준선**입니다.
ARMS = [
    ("m2.5(cnx릴리스)", "stage2_convnextv2_base_m2.5_384_n121k_moderate", "m2.5"),
    ("m2.5(eff)",      "stage2_effnetv2_s_m2.5_384_moderate",            "m2.5"),
    ("f320(eff)",      "stage2_effnetv2_s_f320_384_moderate",            "f320"),
]
N_ROBUST = 2000          # STEP 23 과 **같은 표본 수** (다르면 하락폭 비교 금지)
ZOOMS  = [1.0, 0.5, 0.71, 1.41, 2.0]
SHIFTS = [0.10, 0.20]

ck = env.work_root() / "checkpoints"
missing = [e for _, e, _ in ARMS if not (ck / e / "best.pt").exists()]
if missing:
    have = sorted(p.name for p in ck.glob("stage2*") if (p / "best.pt").exists())
    raise SystemExit("[X] 체크포인트가 없습니다: " + str(missing)
                     + "\n    " + str(ck) + " 안에 있는 것:\n      "
                     + "\n      ".join(have or ["(없음)"])
                     + "\n    STEP 23 출력을 Private 데이터셋으로 만들어 붙이세요.")
print("체크포인트 3개 확인:")
for n, e, t in ARMS:
    print(f"  {n:16} {e}  (크롭 {t})")


## 2. 데이터 — 두 태그가 **다 있는** 청크만

STEP 23 과 같은 val 을 봐야 비교가 됩니다.

In [ ]:
df = labels.load(env.work_root() / "manifests" / "manifest_final.parquet")
print(f"{len(df):,}행")

TAGS = sorted({t for _, _, t in ARMS})
have = crop.available_tags()
print("붙어 있는 태그:", have)
if [t for t in TAGS if t not in have]:
    raise SystemExit(f"[X] 태그 없음: {[t for t in TAGS if t not in have]}")

keep = crop.chunks_with_crops(df, TAGS)
if not keep:
    raise SystemExit("[X] 두 태그가 다 있는 청크가 없습니다.")
df = df[df["chunk"].isin(keep)].reset_index(drop=True)
print(f"\n쓸 청크 {keep} — {len(df):,}행")

# 팔마다 자기 크롭의 val 뷰. **행 순서가 같아야** 앙상블이 됩니다.
views = {}
for name, _, tag in ARMS:
    v = stages.to_stage2(crop.switch_tag(df, tag, verbose=False))
    _, va = split.get_fold(v, 0)
    views[name] = va.reset_index(drop=True)
ref = views[ARMS[0][0]]
for name, v in views.items():
    assert len(v) == len(ref), f"{name} 행 수 불일치 {len(v)} vs {len(ref)}"
    assert (v["label"].to_numpy() == ref["label"].to_numpy()).all(), \
        f"{name} 라벨 순서 불일치 — 앙상블 불가"
print(f"val {len(ref):,}행 · 라벨 순서 일치 확인")
print("클래스별:", ref["label"].value_counts().sort_index().to_dict())

# **모든 팔이 같은 2,000장**을 봐야 합니다 (표본이 다르면 하락폭 비교 금지)
idx = np.random.default_rng(0).choice(len(ref), min(N_ROBUST, len(ref)), replace=False)
idx.sort()
sub = {n: v.iloc[idx].reset_index(drop=True) for n, v in views.items()}
print(f"교란 검사 표본 {len(idx):,}장 (모든 팔 공통)")


## 3. 교란 7조건 × 3모델

`ZoomView(1.0)` 이 기준입니다 — 항등이 **아니라** 437px 로 늘렸다가 384 중앙
크롭입니다. 그래서 여기 절대값을 STEP 23·25 의 macro-F1 과 **섞으면 안 됩니다.**
하락률(상대값)만 오갈 수 있습니다.

In [ ]:
import time
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
from torchvision import transforms as T
from src.data import SkinDataset

cfg = CFG(); cfg.img_size = 384

def load_arm(exp):
    m = models.build(train.model_key_from_exp(exp), n_classes=len(CLASSES),
                     pretrained=False)
    sd = torch.load(ck / exp / "best.pt", map_location="cpu", weights_only=False)
    m.load_state_dict(sd.get("model", sd.get("state_dict", sd)), strict=False)
    return m.to(DEV).eval()

M = {n: load_arm(e) for n, e, _ in ARMS}

def logits_of(model, frame, view):
    mean, std = robust._mean_std(model)
    tf = T.Compose([view, T.ToTensor(), T.Normalize(mean, std)])
    dl = DataLoader(SkinDataset(frame, tf, "crop_path", classes=CLASSES),
                    batch_size=64, shuffle=False,
                    num_workers=cfg.resolved_num_workers())
    out, ys = [], []
    with torch.no_grad():
        for x, yy in dl:
            out.append(model(x.to(DEV)).float().cpu()); ys.append(yy)
    return torch.cat(out).numpy(), torch.cat(ys).numpy()

def prob(a):
    e = np.exp(a - a.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)

conds = ([(f"배율 {z}x", robust.ZoomView(cfg.img_size, z)) for z in ZOOMS]
         + [(f"위치 {s}", robust.ShiftView(cfg.img_size, s)) for s in SHIFTS])
names = [n for n, _, _ in ARMS]
res, t0 = {}, time.time()
for label, view in conds:
    lg, yy = {}, None
    for n in names:
        lg[n], y0 = logits_of(M[n], sub[n], view)
        yy = y0 if yy is None else yy
        assert (y0 == yy).all(), f"{n} 라벨 불일치"
    r = {n: f1_score(yy, lg[n].argmax(1), average="macro", zero_division=0) for n in names}
    r["앙상블"] = f1_score(yy, np.mean([prob(lg[n]) for n in names], 0).argmax(1),
                         average="macro", zero_division=0)
    res[label] = r
    print(f"  {label:10} " + "  ".join(f"{k} {v:.4f}" for k, v in r.items())
          + f"   ({time.time()-t0:.0f}s)")


## 4. 판정 — 사전등록 관문 (`experiments.stage2_ensemble_report`)

In [ ]:
import json
from src import experiments

cols = names + ["앙상블"]
b = res["배율 1.0x"]
worst = {c: min(v[c] for v in res.values()) for c in cols}
drop = {c: (b[c] - worst[c]) / max(b[c], 1e-9) for c in cols}

print(f"\n{'':14}" + "".join(f"{c:>17}" for c in cols))
print(f"{'기준(1.0x)':14}" + "".join(f"{b[c]:>17.4f}" for c in cols))
print(f"{'최악':14}" + "".join(f"{worst[c]:>17.4f}" for c in cols))
print(f"{'하락률':14}" + "".join(f"{drop[c]:>16.1%} " for c in cols))

base_name = names[0]
print(f"\n앙상블 하락 {drop['앙상블']:.1%} vs 기준선({base_name}) {drop[base_name]:.1%}"
      f"  →  {drop['앙상블']-drop[base_name]:+.1%}p")
if drop["앙상블"] <= max(drop[c] for c in names):
    print("  → 최악의 팔보다 낫습니다.")

# STEP 25 가 이미 잰 값 (전체 val 27,331행, 저장된 로짓)
STEP25 = {"base_macro_f1": 0.5959, "ens_macro_f1": 0.6254,
          "d_ci": [0.0243, 0.0346],
          "base_recall": {"A1": .619, "A2": .756, "A3": .761,
                          "A4": .228, "A5": .500, "A6": .628},
          "ens_recall": {"A1": .637, "A2": .782, "A3": .770,
                         "A4": .262, "A5": .541, "A6": .675},
          "base_f2rest": 0.377, "ens_f2rest": 0.370}
verdict = experiments.stage2_ensemble_report(
    {"macro_f1": STEP25["base_macro_f1"], "scale_drop": drop[base_name],
     "recall": STEP25["base_recall"], "focus_to_other": 0.351,
     "focus_to_rest": STEP25["base_f2rest"]},
    {"macro_f1": STEP25["ens_macro_f1"], "d_macro_f1_ci": STEP25["d_ci"],
     "scale_drop": drop["앙상블"], "recall": STEP25["ens_recall"],
     "focus_to_other": 0.368, "focus_to_rest": STEP25["ens_f2rest"]})

out = {"step": "STEP 26 — 2단계 앙상블 배율 강건성 (전체 데이터)",
       "chunks": keep, "n_val": int(len(ref)), "n_robust": int(len(idx)),
       "arms": [{"name": n, "exp": e, "tag": t} for n, e, t in ARMS],
       "conditions": res, "baseline_1x": b, "worst": worst, "drop": drop,
       "verdict": verdict}
p = Path("/kaggle/working/step26_ensemble_robust_full.json")
p.write_text(json.dumps(out, indent=2, ensure_ascii=False, default=float),
             encoding="utf-8")
print(f"\n저장: {p}  ← 이 파일을 공유하세요")
print("\n⚠️ macro-F1 절대값은 STEP 23·25 와 **섞지 마세요** — 기준 조건이"
      " ZoomView(1.0)(437px→384 중앙크롭)이라 평가 파이프라인이 다릅니다."
      " 하락률만 비교 가능합니다.")


## 다음

| 결과 | 뜻 |
|---|---|
| 앙상블 하락 ≤ 기준선 + 12%p | **관문 4개 전부 통과** — 채택 가능. 다만 추론 3배 값을 치를지는 "이름을 말할 것인가" 판단이 먼저 |
| 12%p 넘게 나빠짐 | **기각.** `f320` 의 배율 약점을 물려받은 것 — macro-F1 이 올라도 촬영 거리에 무너집니다 |

어느 쪽이든 `docs/results/STEP26_*.md` 에 남기고 CLAUDE.md 결론표를 갱신합니다.